# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [8]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [15]:
# Prelude: load cleaned data (or rebuild Milestone 1 cleaning), split, and scale
from dataclasses import dataclass
from sklearn.metrics import mean_absolute_error


@dataclass
class RunConfig:
    random_state: int = 42
    fast_mode: bool = True
    cv_splits: int = 2
    cv_repeats: int = 1
    cv_sample_size: int = 6000
    feature_search_sample_size: int = 5000
    tuning_sample_size: int = 5000


CONFIG = RunConfig(random_state=random_state)


def maybe_sample(X_data, y_data, max_rows: int, seed: int):
    """Downsample consistently for fast, repeatable experimentation."""
    if max_rows is None or len(X_data) <= max_rows:
        return X_data, y_data
    sampled_idx = X_data.sample(n=max_rows, random_state=seed).index
    return X_data.loc[sampled_idx], y_data.loc[sampled_idx]


def clean_zillow_like_m1(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Reproduce the Milestone 1 cleaning pipeline so this notebook is self-contained."""
    df_work = raw_df.copy()

    manual_drop_cols = ["parcelid", "rawcensustractandblock", "censustractandblock", "assessmentyear"]
    df_work = df_work.drop(columns=[c for c in manual_drop_cols if c in df_work.columns], errors="ignore")

    missing_pct = df_work.isnull().mean() * 100
    high_null_cols = missing_pct[missing_pct > 95.0].index.tolist()
    df_work = df_work.drop(columns=high_null_cols, errors="ignore")

    target_col_local = "taxvaluedollarcnt"
    df_work = df_work.loc[df_work[target_col_local].notna()].copy()

    row_null_fraction = df_work.isnull().mean(axis=1)
    df_work = df_work.loc[row_null_fraction <= 0.40].copy()

    outlier_cutoff = df_work[target_col_local].quantile(0.995)
    df_work = df_work.loc[df_work[target_col_local] <= outlier_cutoff].copy()

    cat_cols = df_work.select_dtypes(include=["object", "str", "category", "bool"]).columns.tolist()
    num_cols = [c for c in df_work.columns if c not in cat_cols]

    if num_cols:
        num_imputer = SimpleImputer(strategy="median")
        df_work[num_cols] = num_imputer.fit_transform(df_work[num_cols])

    if cat_cols:
        cat_imputer = SimpleImputer(strategy="most_frequent")
        df_work[cat_cols] = cat_imputer.fit_transform(df_work[cat_cols])

        encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            encoded_missing_value=-1,
        )
        df_work[cat_cols] = encoder.fit_transform(df_work[cat_cols])

    return df_work


def build_baseline_models(seed: int):
    """Fast baseline registry: linear + regularized + boosted tree."""
    return {
        "Linear Regression": LinearRegression(),
        "Ridge": Ridge(random_state=seed),
        "Gradient Boosting": GradientBoostingRegressor(random_state=seed),
    }


def evaluate_models_cv(model_dict, X_data, y_data, cv_obj, sample_size=None, seed=42):
    """Evaluate models with repeated CV and return MAE summary and runtime."""
    X_eval, y_eval = maybe_sample(X_data, y_data, sample_size, seed)
    rows = []

    for model_name, model in model_dict.items():
        t0 = time.time()
        cv_scores = cross_val_score(
            model,
            X_eval,
            y_eval,
            scoring="neg_mean_absolute_error",
            cv=cv_obj,
            n_jobs=-1,
        )
        elapsed = time.time() - t0
        mae_scores = -cv_scores
        rows.append(
            {
                "model": model_name,
                "cv_mae_mean": mae_scores.mean(),
                "cv_mae_std": mae_scores.std(),
                "cv_seconds": elapsed,
                "cv_rows_used": len(X_eval),
            }
        )

    return pd.DataFrame(rows).sort_values("cv_mae_mean").reset_index(drop=True)


def add_holdout_metrics(results_df, model_dict, X_train_data, y_train_data, X_test_data, y_test_data):
    """Attach hold-out MAE/RMSE to an existing results table."""
    test_mae = {}
    test_rmse = {}

    for model_name, model in model_dict.items():
        model.fit(X_train_data, y_train_data)
        preds = model.predict(X_test_data)
        test_mae[model_name] = mean_absolute_error(y_test_data, preds)
        test_rmse[model_name] = math.sqrt(mean_squared_error(y_test_data, preds))

    out_df = results_df.copy()
    out_df["test_mae"] = out_df["model"].map(test_mae)
    out_df["test_rmse"] = out_df["model"].map(test_rmse)
    return out_df


cleaned_path = "zillow_cleaned.csv"
if os.path.exists(cleaned_path):
    df = pd.read_csv(cleaned_path)
    print(f"Loaded cleaned dataset from {cleaned_path}")
else:
    print("zillow_cleaned.csv not found. Rebuilding cleaned dataset from zillow_dataset.csv...")
    raw_df = pd.read_csv("zillow_dataset.csv")
    df = clean_zillow_like_m1(raw_df)
    df.to_csv(cleaned_path, index=False)
    print(f"Saved rebuilt cleaned dataset to {cleaned_path}")

print("Final modeling dataset shape:", df.shape)
print("Remaining nulls:", int(df.isnull().sum().sum()))

target_col = "taxvaluedollarcnt"
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=CONFIG.random_state
)

base_scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    base_scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_scaled = pd.DataFrame(
    base_scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index,
)

rkf = RepeatedKFold(
    n_splits=CONFIG.cv_splits,
    n_repeats=CONFIG.cv_repeats,
    random_state=CONFIG.random_state,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print(f"Fast mode: {CONFIG.fast_mode} | CV: {CONFIG.cv_splits}x{CONFIG.cv_repeats} | CV sample: {CONFIG.cv_sample_size}")

Loaded cleaned dataset from zillow_cleaned.csv
Final modeling dataset shape: (74039, 33)
Remaining nulls: 0
Train shape: (59231, 32)
Test shape: (14808, 32)
Fast mode: True | CV: 2x1 | CV sample: 6000


### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [16]:
# Part 1: baseline models with default hyperparameters (fast, configurable execution)
baseline_models = build_baseline_models(CONFIG.random_state)

part1_t0 = time.time()
baseline_results = evaluate_models_cv(
    baseline_models,
    X_train_scaled,
    y_train,
    rkf,
    sample_size=CONFIG.cv_sample_size,
    seed=CONFIG.random_state,
)
baseline_results = add_holdout_metrics(
    baseline_results,
    baseline_models,
    X_train_scaled,
    y_train,
    X_test_scaled,
    y_test,
)
part1_elapsed = time.time() - part1_t0

display(baseline_results.round(2))

best_baseline_model = baseline_results.sort_values("cv_mae_mean").iloc[0]
most_stable_baseline = baseline_results.sort_values("cv_mae_std").iloc[0]

print("Best baseline model by CV MAE:", best_baseline_model["model"])
print("Most stable baseline model (lowest CV std):", most_stable_baseline["model"])
print(f"Part 1 runtime: {part1_elapsed:.1f} seconds")

,model,cv_mae_mean,cv_mae_std,cv_seconds,cv_rows_used,test_mae,test_rmse
0,Gradient Boosting,183685.60,4157.41,1.14,6000,182649.32,284638.97
1,Ridge,209771.37,3565.34,0.03,6000,209804.16,323083.51
2,Linear Regression,210630.33,4392.20,0.02,6000,209805.09,323083.33


Best baseline model by CV MAE: Gradient Boosting
Most stable baseline model (lowest CV std): Ridge
Part 1 runtime: 18.4 seconds


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

> Baseline comparison (Part 1):
- **Best overall model:** Gradient Boosting had the lowest CV MAE (**183,685.60**) and also the best held-out test MAE (**182,649.32**), so it was strongest on both validation and test data.
- **Most stable model:** Ridge had the lowest CV standard deviation (**3,565.34**), indicating the most consistent fold-to-fold behavior.
- **Overfitting/underfitting signals:**
  - Linear Regression and Ridge had much higher MAE than Gradient Boosting, suggesting underfitting relative to the non-linear structure in this dataset.
  - Gradient Boosting did not show obvious overfitting at baseline because CV MAE and test MAE were close in scale, which suggests reasonable generalization.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [17]:
# Part 2: add engineered features and re-evaluate

def add_engineered_features(X_in: pd.DataFrame) -> pd.DataFrame:
    X_out = X_in.copy()

    if "calculatedfinishedsquarefeet" in X_out.columns:
        X_out["log_finishedsquarefeet"] = np.log1p(X_out["calculatedfinishedsquarefeet"].clip(lower=0))

    if "lotsizesquarefeet" in X_out.columns:
        X_out["log_lotsizesquarefeet"] = np.log1p(X_out["lotsizesquarefeet"].clip(lower=0))

    if "yearbuilt" in X_out.columns:
        X_out["home_age"] = 2016 - X_out["yearbuilt"]

    if "bathroomcnt" in X_out.columns and "bedroomcnt" in X_out.columns:
        X_out["bath_to_bed_ratio"] = np.where(
            X_out["bedroomcnt"] > 0,
            X_out["bathroomcnt"] / X_out["bedroomcnt"],
            X_out["bathroomcnt"],
        )

    if "calculatedfinishedsquarefeet" in X_out.columns and "buildingqualitytypeid" in X_out.columns:
        X_out["size_x_quality"] = X_out["calculatedfinishedsquarefeet"] * X_out["buildingqualitytypeid"]

    return X_out


part2_t0 = time.time()

X_train_fe = add_engineered_features(X_train)
X_test_fe = add_engineered_features(X_test)
engineered_features_added = [c for c in X_train_fe.columns if c not in X_train.columns]

fe_scaler = StandardScaler()
X_train_fe_scaled = pd.DataFrame(
    fe_scaler.fit_transform(X_train_fe),
    columns=X_train_fe.columns,
    index=X_train_fe.index,
)
X_test_fe_scaled = pd.DataFrame(
    fe_scaler.transform(X_test_fe),
    columns=X_test_fe.columns,
    index=X_test_fe.index,
)

part2_results = evaluate_models_cv(
    baseline_models,
    X_train_fe_scaled,
    y_train,
    rkf,
    sample_size=CONFIG.cv_sample_size,
    seed=CONFIG.random_state,
)
part2_results = add_holdout_metrics(
    part2_results,
    baseline_models,
    X_train_fe_scaled,
    y_train,
    X_test_fe_scaled,
    y_test,
)

comparison_part2 = baseline_results[["model", "cv_mae_mean", "cv_mae_std"]].merge(
    part2_results[["model", "cv_mae_mean", "cv_mae_std"]],
    on="model",
    suffixes=("_baseline", "_engineered"),
)
comparison_part2["delta_mae"] = (
    comparison_part2["cv_mae_mean_engineered"] - comparison_part2["cv_mae_mean_baseline"]
)

part2_elapsed = time.time() - part2_t0

print("Engineered features added:", engineered_features_added)
print("\nBaseline vs Engineered comparison (negative delta means improvement):")
display(comparison_part2.sort_values("delta_mae").round(2))

print("Part 2 model performance with engineered features:")
display(part2_results.round(2))
print(f"Part 2 runtime: {part2_elapsed:.1f} seconds")

Engineered features added: ['log_finishedsquarefeet', 'log_lotsizesquarefeet', 'home_age', 'bath_to_bed_ratio', 'size_x_quality']

Baseline vs Engineered comparison (negative delta means improvement):


,model,cv_mae_mean_baseline,cv_mae_std_baseline,cv_mae_mean_engineered,cv_mae_std_engineered,delta_mae
2,Linear Regression,210630.33,4392.20,208564.39,3541.26,-2065.93
1,Ridge,209771.37,3565.34,207733.59,2792.40,-2037.78
0,Gradient Boosting,183685.60,4157.41,182747.84,2810.47,-937.76


Part 2 model performance with engineered features:


,model,cv_mae_mean,cv_mae_std,cv_seconds,cv_rows_used,test_mae,test_rmse
0,Gradient Boosting,182747.84,2810.47,1.57,6000,182177.22,283858.43
1,Ridge,207733.59,2792.40,0.02,6000,208074.18,322155.96
2,Linear Regression,208564.39,3541.26,0.12,6000,208075.26,322155.54


Part 2 runtime: 26.0 seconds


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




> Feature engineering impact (Part 2):
- **Did performance improve?** Yes, all three models improved in CV MAE after adding engineered features.
  - Linear Regression: **-2,065.93** MAE (improved)
  - Ridge: **-2,037.78** MAE (improved)
  - Gradient Boosting: **-937.76** MAE (improved)
- **Which features seemed to help most?**
  - The strongest contributors appear to be scale/structure features such as `log_finishedsquarefeet`, `size_x_quality`, and `bath_to_bed_ratio`, which add non-linear and interaction-style information.
- **Why these helped:**
  - Log transforms reduce skew for size-related variables.
  - Interaction (`size_x_quality`) captures that area and build quality together are more informative than either alone.
  - Ratio (`bath_to_bed_ratio`) provides layout efficiency signal that raw counts alone may miss.
- **Model-specific note:** Linear and Ridge benefited more in relative terms, which is expected because engineered features help linear models represent non-linear patterns indirectly.

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [18]:
# Part 3: feature selection and re-evaluation (compact search for speed)
part3_t0 = time.time()

cv_fast = RepeatedKFold(n_splits=3, n_repeats=1, random_state=CONFIG.random_state)
X_search, y_search = maybe_sample(
    X_train_fe,
    y_train,
    CONFIG.feature_search_sample_size,
    CONFIG.random_state,
)

max_k = X_train_fe.shape[1]
k_candidates = [k for k in [8, 12, 16, max_k] if 1 <= k <= max_k]
k_candidates = sorted(set(k_candidates))

part3_rows = []
part3_best_features = {}

for model_name, model in baseline_models.items():
    best_k = k_candidates[0]
    best_score = np.inf

    for k in k_candidates:
        selector = SelectKBest(score_func=f_regression, k=k)
        selected_cols = X_search.columns[selector.fit(X_search, y_search).get_support()].tolist()

        X_search_sel = X_search[selected_cols]
        search_scaler = StandardScaler()
        X_search_sel_scaled = pd.DataFrame(
            search_scaler.fit_transform(X_search_sel),
            columns=selected_cols,
            index=X_search_sel.index,
        )

        score_df = evaluate_models_cv(
            {model_name: model},
            X_search_sel_scaled,
            y_search,
            cv_fast,
            sample_size=None,
            seed=CONFIG.random_state,
        )
        score = score_df.loc[0, "cv_mae_mean"]

        if score < best_score:
            best_score = score
            best_k = k

    final_selector = SelectKBest(score_func=f_regression, k=best_k)
    selected_features = X_train_fe.columns[final_selector.fit(X_train_fe, y_train).get_support()].tolist()
    part3_best_features[model_name] = selected_features

    X_train_sel = X_train_fe[selected_features].copy()
    X_test_sel = X_test_fe[selected_features].copy()

    final_scaler = StandardScaler()
    X_train_sel_scaled = pd.DataFrame(
        final_scaler.fit_transform(X_train_sel),
        columns=selected_features,
        index=X_train_sel.index,
    )
    X_test_sel_scaled = pd.DataFrame(
        final_scaler.transform(X_test_sel),
        columns=selected_features,
        index=X_test_sel.index,
    )

    cv_summary = evaluate_models_cv(
        {model_name: model},
        X_train_sel_scaled,
        y_train,
        rkf,
        sample_size=CONFIG.cv_sample_size,
        seed=CONFIG.random_state,
    ).iloc[0]

    model.fit(X_train_sel_scaled, y_train)
    preds = model.predict(X_test_sel_scaled)

    part3_rows.append(
        {
            "model": model_name,
            "n_features_selected": best_k,
            "selected_features": selected_features,
            "cv_mae_mean": cv_summary["cv_mae_mean"],
            "cv_mae_std": cv_summary["cv_mae_std"],
            "test_mae": mean_absolute_error(y_test, preds),
            "test_rmse": math.sqrt(mean_squared_error(y_test, preds)),
        }
    )

part3_results = pd.DataFrame(part3_rows).sort_values("cv_mae_mean").reset_index(drop=True)
part3_elapsed = time.time() - part3_t0

display(part3_results[["model", "n_features_selected", "cv_mae_mean", "cv_mae_std", "test_mae", "test_rmse"]].round(2))

print("\nSelected feature subsets by model:")
for model_name, feats in part3_best_features.items():
    preview = feats[:10]
    suffix = " ..." if len(feats) > 10 else ""
    print(f"{model_name} ({len(feats)} features): {preview}{suffix}")

print(f"Part 3 runtime: {part3_elapsed:.1f} seconds")

,model,n_features_selected,cv_mae_mean,cv_mae_std,test_mae,test_rmse
0,Gradient Boosting,37,182747.84,2810.47,182177.22,283858.43
1,Ridge,37,207733.59,2792.40,208074.18,322155.96
2,Linear Regression,37,208564.39,3541.26,208075.26,322155.54



Selected feature subsets by model:
Linear Regression (37 features): ['airconditioningtypeid', 'bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedbathnbr', 'finishedfloor1squarefeet', 'calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'finishedsquarefeet50', 'fips'] ...
Ridge (37 features): ['airconditioningtypeid', 'bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedbathnbr', 'finishedfloor1squarefeet', 'calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'finishedsquarefeet50', 'fips'] ...
Gradient Boosting (37 features): ['airconditioningtypeid', 'bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedbathnbr', 'finishedfloor1squarefeet', 'calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'finishedsquarefeet50', 'fips'] ...
Part 3 runtime: 32.7 seconds


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


> Feature selection analysis (Part 3):
- **Did performance improve after feature selection?** Not meaningfully in this run. The selected subset size converged to **37 features** (the full engineered set) for each model, and the MAE values were essentially unchanged from Part 2.
- **Which features were consistently retained?** Core property size, bathroom/bedroom, quality, and location features were retained across models (for example `calculatedfinishedsquarefeet`, `bathroomcnt`, `bedroomcnt`, `buildingqualitytypeid`, and location IDs).
- **Were engineered features selected?** Yes. Engineered variables such as `log_finishedsquarefeet`, `log_lotsizesquarefeet`, `home_age`, `bath_to_bed_ratio`, and `size_x_quality` were included in the selected feature set.
- **Interpretation:** The result suggests most features in the engineered dataset still carried signal, so aggressive pruning did not provide additional gain under the fast configuration.

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [21]:
# Part 4: hyperparameter tuning for the three selected models (runtime-capped)
part4_t0 = time.time()

part4_rows = []
tuned_artifacts = {}

for model_name in baseline_models.keys():
    selected_features = part3_best_features[model_name]

    X_train_model = X_train_fe[selected_features].copy()
    X_test_model = X_test_fe[selected_features].copy()

    model_scaler = StandardScaler()
    X_train_model_scaled = pd.DataFrame(
        model_scaler.fit_transform(X_train_model),
        columns=selected_features,
        index=X_train_model.index,
    )
    X_test_model_scaled = pd.DataFrame(
        model_scaler.transform(X_test_model),
        columns=selected_features,
        index=X_test_model.index,
    )

    X_tune, y_tune = maybe_sample(
        X_train_model_scaled,
        y_train,
        CONFIG.tuning_sample_size,
        CONFIG.random_state,
    )

    if model_name == "Linear Regression":
        tuner = GridSearchCV(
            estimator=LinearRegression(),
            param_grid={"fit_intercept": [True, False]},
            scoring="neg_mean_absolute_error",
            cv=2,
            n_jobs=-1,
        )
    elif model_name == "Ridge":
        tuner = GridSearchCV(
            estimator=Ridge(random_state=CONFIG.random_state),
            param_grid={"alpha": [0.1, 1.0, 10.0, 30.0, 100.0]},
            scoring="neg_mean_absolute_error",
            cv=2,
            n_jobs=-1,
        )
    else:
        tuner = RandomizedSearchCV(
            estimator=GradientBoostingRegressor(random_state=CONFIG.random_state),
            param_distributions={
                "n_estimators": [80, 120, 180],
                "learning_rate": [0.03, 0.05, 0.1],
                "max_depth": [2, 3, 4],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4],
                "subsample": [0.7, 0.9, 1.0],
            },
            n_iter=4,
            scoring="neg_mean_absolute_error",
            cv=2,
            random_state=CONFIG.random_state,
            n_jobs=-1,
        )

    tuner.fit(X_tune, y_tune)
    best_model = tuner.best_estimator_

    tuned_cv = evaluate_models_cv(
        {model_name: best_model},
        X_train_model_scaled,
        y_train,
        rkf,
        sample_size=CONFIG.cv_sample_size,
        seed=CONFIG.random_state,
    ).iloc[0]

    best_model.fit(X_train_model_scaled, y_train)
    preds = best_model.predict(X_test_model_scaled)

    tuned_artifacts[model_name] = {
        "model": best_model,
        "features": selected_features,
        "scaler": model_scaler,
    }

    part4_rows.append(
        {
            "model": model_name,
            "best_params": tuner.best_params_,
            "cv_mae_mean": tuned_cv["cv_mae_mean"],
            "cv_mae_std": tuned_cv["cv_mae_std"],
            "test_mae": mean_absolute_error(y_test, preds),
            "test_rmse": math.sqrt(mean_squared_error(y_test, preds)),
        }
    )

part4_results = pd.DataFrame(part4_rows).sort_values("cv_mae_mean").reset_index(drop=True)
part4_elapsed = time.time() - part4_t0

display(part4_results[["model", "cv_mae_mean", "cv_mae_std", "test_mae", "test_rmse"]].round(2))

print("\nBest hyperparameters by model:")
for _, row in part4_results.iterrows():
    print(f"{row['model']}: {row['best_params']}")

print(f"Part 4 runtime: {part4_elapsed:.1f} seconds")

,model,cv_mae_mean,cv_mae_std,test_mae,test_rmse
0,Gradient Boosting,181394.67,2948.21,181617.52,283150.05
1,Ridge,202408.67,1968.13,207986.70,322186.66
2,Linear Regression,208564.39,3541.26,208075.26,322155.54



Best hyperparameters by model:
Gradient Boosting: {'subsample': 0.7, 'n_estimators': 120, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 4, 'learning_rate': 0.05}
Ridge: {'alpha': 100.0}
Linear Regression: {'fit_intercept': True}
Part 4 runtime: 36.5 seconds


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Tuning strategy and outcomes (Part 4):
- **Strategy by model:**
  - Linear Regression: small grid on `fit_intercept` (minimal but valid check).
  - Ridge: grid search over `alpha` values to control regularization strength.
  - Gradient Boosting: randomized search over tree depth, number of estimators, learning rate, subsample, and split/leaf controls.
- **Best hyperparameters found:**
  - Gradient Boosting: `subsample=0.7`, `n_estimators=120`, `min_samples_split=2`, `min_samples_leaf=2`, `max_depth=4`, `learning_rate=0.05`
  - Ridge: `alpha=100.0`
  - Linear Regression: `fit_intercept=True`
- **Did tuning help?**
  - Yes, especially for Gradient Boosting: CV MAE improved from **182,747.84** (Part 3) to **181,394.67**.
  - Ridge also improved from **207,733.59** to **202,408.67** after stronger regularization.
- **Preprocessing interactions:**
  - Engineered features and scaling were most beneficial for linear models (Linear/Ridge), while Gradient Boosting remained the strongest overall due to its ability to capture non-linear relationships directly.

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [22]:
# Part 5: choose final model and report final CV + held-out test performance
part5_t0 = time.time()

final_row = part4_results.sort_values("cv_mae_mean").iloc[0]
final_model_name = final_row["model"]
final_model = tuned_artifacts[final_model_name]["model"]
final_features = tuned_artifacts[final_model_name]["features"]

X_train_final = X_train_fe[final_features].copy()
X_test_final = X_test_fe[final_features].copy()

final_scaler = StandardScaler()
X_train_final_scaled = pd.DataFrame(
    final_scaler.fit_transform(X_train_final),
    columns=final_features,
    index=X_train_final.index,
)
X_test_final_scaled = pd.DataFrame(
    final_scaler.transform(X_test_final),
    columns=final_features,
    index=X_test_final.index,
)

final_cv = evaluate_models_cv(
    {final_model_name: final_model},
    X_train_final_scaled,
    y_train,
    rkf,
    sample_size=CONFIG.cv_sample_size,
    seed=CONFIG.random_state,
).iloc[0]

final_model.fit(X_train_final_scaled, y_train)
final_preds = final_model.predict(X_test_final_scaled)

final_test_mae = mean_absolute_error(y_test, final_preds)
final_test_rmse = math.sqrt(mean_squared_error(y_test, final_preds))
part5_elapsed = time.time() - part5_t0

print("Final selected model:", final_model_name)
print("Number of features used:", len(final_features))
print("CV MAE mean:", round(final_cv["cv_mae_mean"], 2))
print("CV MAE std:", round(final_cv["cv_mae_std"], 2))
print("Held-out test MAE:", round(final_test_mae, 2))
print("Held-out test RMSE:", round(final_test_rmse, 2))
print(f"Part 5 runtime: {part5_elapsed:.1f} seconds")

final_summary = pd.DataFrame(
    [
        {
            "final_model": final_model_name,
            "n_features": len(final_features),
            "cv_rows_used": final_cv["cv_rows_used"],
            "cv_mae_mean": final_cv["cv_mae_mean"],
            "cv_mae_std": final_cv["cv_mae_std"],
            "test_mae": final_test_mae,
            "test_rmse": final_test_rmse,
        }
    ]
)

display(final_summary.round(2))
print("Final feature set:")
print(final_features)

Final selected model: Gradient Boosting
Number of features used: 37
CV MAE mean: 181394.67
CV MAE std: 2948.21
Held-out test MAE: 181617.52
Held-out test RMSE: 283150.05
Part 5 runtime: 27.9 seconds


,final_model,n_features,cv_rows_used,cv_mae_mean,cv_mae_std,test_mae,test_rmse
0,Gradient Boosting,37,6000,181394.67,2948.21,181617.52,283150.05


Final feature set:
['airconditioningtypeid', 'bathroomcnt', 'bedroomcnt', 'buildingqualitytypeid', 'calculatedbathnbr', 'finishedfloor1squarefeet', 'calculatedfinishedsquarefeet', 'finishedsquarefeet12', 'finishedsquarefeet50', 'fips', 'fireplacecnt', 'fullbathcnt', 'garagecarcnt', 'garagetotalsqft', 'heatingorsystemtypeid', 'latitude', 'longitude', 'lotsizesquarefeet', 'poolcnt', 'pooltypeid7', 'propertycountylandusecode', 'propertylandusetypeid', 'propertyzoningdesc', 'regionidcity', 'regionidcounty', 'regionidneighborhood', 'regionidzip', 'roomcnt', 'threequarterbathnbr', 'unitcnt', 'yearbuilt', 'numberofstories', 'log_finishedsquarefeet', 'log_lotsizesquarefeet', 'home_age', 'bath_to_bed_ratio', 'size_x_quality']


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> 1. Model Selection
- **Final model:** Gradient Boosting.
- **Why:** It achieved the best tuned validation and test performance in this notebook run.
  - CV MAE mean: **181,394.67**
  - CV MAE std: **2,948.21**
  - Held-out test MAE: **181,617.52**
  - Held-out test RMSE: **283,150.05**
- **Trade-offs:** Gradient Boosting is less interpretable than Linear/Ridge, but the performance gain justified choosing it as the final model.

2. Revisiting an Early Decision
- **Decision revisited:** Adding engineered features from Milestone 1 (`log_finishedsquarefeet`, `log_lotsizesquarefeet`, `home_age`, `bath_to_bed_ratio`, `size_x_quality`).
- **Original rationale:** Capture non-linear scale effects, interaction effects, and layout-efficiency signals that raw columns might miss.
- **What happened in the full pipeline:** This step helped performance in Part 2 for all three models (all had lower CV MAE after engineering), so the decision was kept.
- **Evidence:** Part 2 MAE deltas vs baseline were all negative, including **-937.76** for Gradient Boosting and around **-2k** for Linear/Ridge.

3. Lessons Learned
- Core value signal comes from property size, room/bath capacity, quality, and location-related fields.
- Feature engineering is useful even when tree models are strong, and especially useful for linear models.
- Hyperparameter tuning produced meaningful additional gain, especially for Gradient Boosting.
- If more time/data were available, next steps would include:
  - richer spatial features (true geo-clusters or external neighborhood data),
  - robust outlier-sensitive loss experiments,
  - deeper model comparison (e.g., XGBoost/LightGBM) under the same fast and full-evaluation settings.